# Gradient Descent

경사하강법(Gradient Descent)은 **손실 함수가 작아지는 방향으로 파라미터를 반복해서 수정하는 학습 방법**이다.

<img src="https://d.pr/i/0vERpX+" alt="경사하강법" width="500">

| 항목 | 설명 |
|---|---|
| 목적 | 손실 함수가 가장 작아지는 파라미터를 찾는 것 |
| 방식 | 손실 함수의 기울기, 즉 gradient의 반대 방향으로 파라미터를 업데이트함 |
| 반복 | 한 번에 정답을 찾는 것이 아니라 여러 번 조금씩 수정함 |

## 선형 회귀에서 예측식

$$
\hat{y} = w_0 + w_1x
$$

- `x`: 입력 feature 값.
- `y`: 실제 정답 값.
- `ŷ`: 모델이 예측한 값.
- `w0`: 절편. 그래프를 위아래로 이동시키는 값.
- `w1`: 기울기. x가 증가할 때 y가 얼마나 증가하는지 정하는 값.
- 모델 학습: 데이터에 잘 맞는 `w0`, `w1`을 찾는 과정.

## 손실 함수가 필요한 이유

모델이 잘 맞는지 판단하려면 "잘 맞음"을 숫자로 바꿔야 함.  
이 숫자가  **손실 함수(Loss Function)** 이다.

```text
오차 = 실제값 - 예측값
손실 함수 = 여러 데이터의 오차를 하나의 숫자로 요약한 값
```

손실 값이 크면 모델이 많이 틀린 것이고, 손실 값이 작으면 모델이 데이터에 더 잘 맞는 것이다.

## RSS와 MSE

회귀에서 자주 사용하는 손실 계산 방식에는 RSS와 MSE가 있다.

| 용어 | 의미 | 특징 |
|---|---|---|
| RSS | 오차 제곱합 | 모든 오차 제곱을 더함 |
| MSE | 평균 제곱 오차 | RSS를 데이터 개수 `N`으로 나눈 값 |

RSS는 다음과 같음.

$$
RSS = \sum_{i=1}^{N}(y_i - \hat{y}_i)^2
$$

MSE는 다음과 같음.

$$
MSE = \frac{1}{N}\sum_{i=1}^{N}(y_i - \hat{y}_i)^2
$$

 `np.mean(diff ** 2)`를 사용하므로 MSE 기준으로 설명함.

## 편미분이 필요한 이유

경사하강법은 `w0`, `w1`을 조금씩 수정해야 함.  
그런데 아무 방향으로 움직이면 안 되고, **손실이 줄어드는 방향**으로 움직여야 함.

이때 필요한 것이 편미분이다.

| 용어 | 의미 |
|---|---|
| 미분 | 한 변수가 조금 변할 때 결과가 어떻게 변하는지 보는 것 |
| 편미분 | 여러 변수 중 하나만 움직였을 때 결과가 어떻게 변하는지 보는 것 |
| gradient | 각 파라미터에 대한 편미분 값을 모아 놓은 기울기 정보 |

현재 손실 함수는 `w0`, `w1` 두 값에 의해 달라짐.  
그래서 각각 따로 질문해야 함.

```text
w0를 조금 바꾸면 MSE가 어떻게 변하는가?
w1을 조금 바꾸면 MSE가 어떻게 변하는가?
```

이 질문의 답이 `w0_grad`, `w1_grad`이다.

## 업데이트 식

$$
w := w - \eta \cdot \frac{dL(w)}{dw}
$$

- `w`: 수정할 파라미터.
- `L(w)`: 손실 함수.
- `dL(w) / dw`: 현재 위치에서 손실이 증가하는 방향의 기울기.
- `η`: 학습률. 한 번에 움직이는 크기.
- `-`를 붙이는 이유는 손실이 증가하는 방향의 반대로 가야 손실이 줄어들기 때문이다.

## 수업용 한 문장

경사하강법은 손실 함수를 보고 `w0`, `w1`을 어느 방향으로 고쳐야 할지 계산한 뒤, 그 반대 방향으로 조금씩 움직이면서 오차를 줄이는 방법이다.


In [ ]:
# 초기 세팅용 import 구문입니다. 먼저 실행한 뒤 실습 코드를 작성합니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## 예제 데이터 생성

경사하강법을 이해하기 위해 정답 식이 있는 가짜 회귀 데이터를 생성함.

```text
y = 6 * X + 4 + noise
```

여기서 `6`은 실제 기울기, `4`는 실제 절편.  
모델은 이 값을 모르는 상태에서 시작하고, 경사하강법으로 `w1 ≈ 6`, `w0 ≈ 4`에 가까운 값을 찾아가야 함.

- 진행 순서
    - 처음에는 아무것도 모르는 상태에서 `w0=0`, `w1=0`으로 시작함.
    - 예측값과 실제값의 차이를 손실로 계산함.
    - 손실이 줄어드는 방향으로 `w0`, `w1`을 반복 수정함.


## 오차, 손실 함수, 파라미터 업데이트 1회

경사하강법 전체를 반복하기 전에, 한 번의 업데이트가 어떻게 일어나는지 직접 계산함.

```text
1. 현재 w0, w1로 y_pred 계산
2. 실제값 y와 예측값 y_pred의 차이 diff 계산
3. diff를 이용해 손실 함수 MSE 계산
4. MSE를 w0, w1로 각각 편미분해 gradient 계산
5. gradient의 반대 방향으로 w0, w1 업데이트
```

처음에는 `w0=0`, `w1=0`에서 시작하므로 예측선이 데이터와 잘 맞지 않음.  
업데이트를 한 번 수행하면 손실이 아주 조금 줄어드는 방향으로 값이 이동함.

## 이 코드에서 쓰는 공식

예측식은 다음과 같음.

$$
\hat{y}_i = w_0 + w_1x_i
$$

코드에서는 오차를 다음처럼 둠.

$$
diff_i = y_i - \hat{y}_i
$$

MSE는 다음과 같음.

$$
MSE = \frac{1}{N}\sum_{i=1}^{N}diff_i^2
$$

MSE를 `w0`로 편미분하면 다음과 같음.

$$
\frac{\partial MSE}{\partial w_0} = -\frac{2}{N}\sum_{i=1}^{N}diff_i
$$

MSE를 `w1`로 편미분하면 다음과 같음.

$$
\frac{\partial MSE}{\partial w_1} = -\frac{2}{N}\sum_{i=1}^{N}x_i \cdot diff_i
$$

위 공식을 코드의 변환하면 다음과 같음.

```python
w0_grad = (-2 / N) * np.sum(diff)
w1_grad = (-2 / N) * np.dot(X.T, diff)
```

이후 학습률 `lr`을 곱해서 w0, w1을 업데이트함.

```python
w0 -= lr * w0_grad
w1 -= lr * w1_grad
```


## Batch Gradient Descent

Batch Gradient Descent는 **전체 훈련 데이터**를 사용해 한 번의 업데이트 방향을 계산함.

한 번 업데이트할 때 전체 데이터를 모두 보기 때문에, 기울기 방향이 비교적 안정적.
다만 데이터가 아주 많으면 매번 전체 데이터를 계산해야 해서 느릴 수 있음.

| 구분 | 의미 |
|---|---|
| 사용하는 데이터 | 전체 데이터 |
| 장점 | 업데이트 방향이 안정적 |
| 단점 | 데이터가 많으면 계산량이 큼 |
| 수업 포인트 | 손실이 반복마다 점점 줄어드는지 확인함 |


## 학습된 `w0`, `w1`로 예측선을 만들고, 실제 데이터와 얼마나 가까운지 확인해보기.

1. 회귀선 그래프: 빨간 선이 실제 데이터의 경향을 잘 따라가는지 확인함.
2. 손실 그래프: 반복이 진행될수록 MSE가 줄어드는지 확인함.

손실 그래프가 빠르게 내려가다가 점점 완만해지면, 모델이 더 이상 크게 고칠 부분이 줄어드는 것으로 해석할 수 있음.


## Mini-batch Gradient Descent

Mini-batch 방식은 전체 데이터가 아니라 **일부 데이터 묶음(batch)** 만 사용해 한 번의 업데이트를 수행함.

전체 데이터를 매번 모두 사용하지 않기 때문에 Batch 방식보다 계산량이 적음.  
데이터 1개만 사용하는 SGD(확률적 경사하강법) 방식보다는 방향이 덜 흔들림.
그래서 실제 딥러닝 학습에서는 Mini-batch 방식이 가장 많이 사용됨.

| 구분 | 의미 |
|---|---|
| 사용하는 데이터 | 일부 데이터 묶음 |
| 장점 | Batch보다 빠르고, SGD보다 안정적 |
| 단점 | batch 선택에 따라 손실이 약간 흔들릴 수 있음 |

주의할 점은 `batch_size`가 업데이트 방향의 안정성에 영향을 준다는 점.  
`batch_size`가 작으면 빠르지만 흔들림이 커지고, 크면 안정적이지만 계산량이 늘어남.


## Stochastic Gradient Descent (확률적 경사 하강법)

Stochastic Gradient Descent는 매번 **데이터 1개**만 사용해 업데이트함.

데이터 1개만 보기 때문에 한 번의 업데이트는 매우 빠름.  
하지만 선택된 데이터 하나에 크게 영향을 받기 때문에 업데이트 방향이 많이 흔들릴 수 있음.

그래서 SGD에서는 다음 두 값이 다를 수 있음.

| 구분 | 의미 |
|---|---|
| 마지막 MSE | 정해진 반복 횟수까지 모두 업데이트한 뒤의 MSE |
| 최소 MSE | 반복 과정 중 가장 낮았던 MSE |

Batch GD는 전체 데이터를 보고 방향을 계산하므로 손실이 비교적 안정적으로 줄어듦.  
반면 SGD는 데이터 1개만 보고 방향을 계산하므로 중간에 좋아졌다가 다시 나빠질 수 있음.

따라서 SGD에서는 필요하면 `best_mse`, `best_w0`, `best_w1`을 따로 저장해 둔다.

세 방식의 차이는 한 번 업데이트할 때 사용하는 데이터 수로 정리할 수 있음.

| 방식 | 한 번 업데이트에 사용하는 데이터 | 특징 |
|---|---:|---|
| Batch GD | 전체 데이터 | 안정적이지만 느릴 수 있음 |
| Mini-batch GD | 일부 묶음 | 속도와 안정성의 절충 |
| Stochastic GD | 1개 | 빠르지만 흔들림이 큼 |


## 최종 정리

- 손실 함수는 모델이 얼마나 틀렸는지 숫자로 계산하는 기준.
- 경사하강법은 손실이 줄어드는 방향으로 파라미터를 반복 수정하는 방법.
- 학습률은 한 번에 움직이는 크기이며, 너무 크거나 작으면 학습이 잘 되지 않을 수 있음.
- Batch GD는 전체 데이터, Mini-batch GD는 일부 묶음, SGD는 데이터 1개로 업데이트함.
- 실제 딥러닝에서는 속도와 안정성의 균형 때문에 Mini-batch 방식이 가장 많이 사용된다.
